# ⛰ Core Lab 8 — Hand a Job to the GPU
**A Billion Small Steps · Hardware for AI · Step 4 · Book sections 4.1–4.3 · Foothold lessons 4.1–4.3**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

**What you should see:** PCIe far slower than GPU memory; copies dominating a single operation; block-size effects; writing past the end without the check.

| Part | What you do | Book |
|---|---|---|
| A | Where tensors live, and how fast PCIe is | 4.1 |
| B | The five steps of offloading, one at a time | 4.2 |
| C | Write your own kernels; what happens without the bounds check | 4.3 |

⏱️ About 2 hours. 📘 Textbook: Hwu, Kirk & El Hajj Ch 2–3 · Hamacher Ch 7. Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

## Part A · Host and Device  *(Section 4.1)*

Every tensor lives on one device. The GPU can only compute on tensors in its own memory.

In [ ]:
# 3 random numbers in CPU (host) memory, the default, and 3 made directly in GPU (device) memory
x_cpu = torch.rand(3)
x_gpu = torch.rand(3, device=device)   # 3 random numbers made directly in GPU memory
# .device tells you where a tensor lives
print("x_cpu lives on:", x_cpu.device)
print("x_gpu lives on:", x_gpu.device)   # and where this one lives

# try mixing the two; if it fails, show the first line of PyTorch's error instead of crashing
try:
    y = x_cpu + x_gpu   # mixing devices: this fails
except RuntimeError as e:   # e = the error PyTorch raised
    print("\nRuntimeError:", str(e).splitlines()[0])   # show the first line of the error message

# the fix: move the CPU tensor to the GPU first
y = x_cpu.to(device) + x_gpu
print("after moving:", y.device)   # now it works, on the GPU

**How fast is PCIe?** Copy tensors of growing size from host to device. Then repeat with **pinned** (page-locked) host memory, which the GPU's DMA engine can read directly. (A T4 is on PCIe Gen 3 ×16: about 16 GB/s on paper, about 12 GB/s in practice.)

In [ ]:
# copy sizes in megabytes, and a table heading
sizes_mb = [1, 10, 100, 500]
print(f"{'size':>8s} {'normal':>12s} {'pinned':>12s}")   # the table heading
for mb in sizes_mb:   # for each size
    # how many 4-byte floats make this many MB; one copy in ordinary memory, one in pinned memory
    n = mb * 1024 * 1024 // 4
    normal = torch.rand(n)   # a tensor in ordinary host memory
    pinned = normal.pin_memory()   # the same numbers in pinned (page-locked) host memory
    # time a host → device copy from each
    t_n = timeit(lambda: normal.to(device), repeats=3)
    t_p = timeit(lambda: pinned.to(device), repeats=3)   # ...and from pinned memory
    # speed = GB ÷ seconds (MB ÷ 1024 = GB)
    print(f"{mb:6d} MB {mb / 1024 / t_n:9.2f} GB/s {mb / 1024 / t_p:9.2f} GB/s")

### ✏️ YOUR TURN (A)
1. What PCIe speed did you measure? How does it compare with the GPU's own memory (measured in Core Lab 7)?
2. Why are small copies (1 MB) so much slower per byte than big ones?

*Your answers:*

## Part B · The Five Steps, One at a Time  *(Section 4.2)*

ReLU on 100 MB of data, with every step as its own line and its own timer. `sync()` makes each timer honest.

In [ ]:
# 100 MB of float32 input data in host memory, and a dictionary of step name → time taken
n = 25_000_000
a_cpu = torch.randn(n)   # the input data, in host memory
steps = {}   # step name → time taken
# warm up: run the ReLU kernel once on a few numbers, so step 3 times the work, not the first call's start-up
torch.clamp(torch.randn(8, device=device), min=0)

# on each line below: do one step, sync() so the GPU really finishes, record the time, restart the clock
sync(); t = time.perf_counter()
# 1 · allocate space for the answer in GPU memory
b_gpu = torch.empty(n, device=device)
sync(); steps["1 allocate"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 1, restart the clock
# 2 · copy the input in, across PCIe
a_gpu = a_cpu.to(device)
sync(); steps["2 copy in"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 2, restart the clock
# 3 · launch the kernel: ReLU, written as "clamp below at 0"
torch.clamp(a_gpu, min=0, out=b_gpu)
sync(); steps["3 kernel"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 3, restart the clock
# 4 · copy the answer out, across PCIe
b_cpu = b_gpu.cpu()
sync(); steps["4 copy out"] = time.perf_counter() - t; t = time.perf_counter()   # wait, record step 4, restart the clock
# 5 · free the GPU memory
del a_gpu, b_gpu
sync(); steps["5 free"] = time.perf_counter() - t   # wait, and record step 5

total = sum(steps.values())   # the time for all five steps together
for k, v in steps.items():   # for each step and its time
    # a text bar chart: up to 40 blocks per step, in proportion to its share of the time
    print(f"{k:11s} {v*1e3:8.2f} ms  {'█' * int(40 * v / total)}")
# compare with ReLU done on the CPU
print("Correct result:", torch.equal(b_cpu, torch.relu(a_cpu)))

The same five steps in **CuPy**. Notice how closely it mirrors the CUDA C version from the lesson (`cudaMalloc`, `cudaMemcpy`…).

In [ ]:
# the same five steps in CuPy, which mirrors CUDA C closely ("d_" = lives on the device)
a_host = np.random.randn(n).astype(np.float32)
# 1 allocate (cudaMalloc)
d_b = cp.empty(n, dtype=cp.float32)
# 2 copy in (cudaMemcpy host to device)
d_a = cp.asarray(a_host)
# 3 launch a kernel
cp.maximum(d_a, 0, out=d_b)
# 4 copy out (cudaMemcpy device to host)
b_host = cp.asnumpy(d_b)
# 5 free (cudaFree, through CuPy's memory pool)
del d_a, d_b
print("CuPy result correct:", np.allclose(b_host, np.maximum(a_host, 0)))   # check against ReLU on the CPU

### ✏️ YOUR TURN (B)
1. Which steps took the most time? What percentage was the actual computing (step 3)?
2. Change the code to run `torch.relu` **50 times** on `a_gpu` before copying out. What happens to the computing share?

*Your answers:*

## Part C · Write Your Own Kernels  *(Section 4.3)*

Here's the ReLU kernel from the lesson, in CUDA C, compiled by CuPy. Each thread computes its global index `i` and handles one element.

In [ ]:
# the ReLU kernel in CUDA C (inside the C code, comments start with //)
relu_src = r'''
// a kernel: runs on the GPU, one copy per thread; x = input, y = output, n = how many elements
extern "C" __global__
void relu(const float *x, float *y, int n) {   // x = input array, y = output array, n = how many
    // this thread's global index
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    // the safety check, then ReLU: the larger of x[i] and 0
    if (i < n)
        y[i] = fmaxf(x[i], 0.0f);   // ReLU: the larger of x[i] and 0
}
'''
# CuPy compiles the C code into a GPU kernel
relu_k = cp.RawKernel(relu_src, "relu")
# deliberately NOT a multiple of any block size; input and output in GPU memory
n = 10_000_003
d_x = cp.random.randn(n, dtype=cp.float32); d_y = cp.empty_like(d_x)   # input and output arrays in GPU memory
block_ms = {}   # block size → time in ms
for threads in [32, 100, 256, 1024]:   # try four block sizes
    # round UP, so every element gets a thread
    blocks = (n + threads - 1) // threads
    # launch: kernel((blocks,), (threads per block,), (arguments)); cp.int32(n) passes n as a C int
    block_ms[threads] = cuda_time_ms(lambda: relu_k((blocks,), (threads,), (d_x, d_y, cp.int32(n))))
    # check against CuPy's own ReLU
    ok = bool(cp.allclose(d_y, cp.maximum(d_x, 0)))
    print(f"block size {threads:5d}: {blocks:7,d} blocks  {block_ms[threads]:6.3f} ms  correct={ok}")   # show the blocks, the time and the check

**What if you forget `if (i < n)`?** Writing past the end of a real array can crash the whole Colab session, so we do it safely: the output array gets 1,024 extra 'sentinel' slots set to −1, and we look at whether the kernel scribbled over them.

In [ ]:
unsafe_src = r'''
extern "C" __global__   // a kernel: runs on the GPU, launched from the CPU
void relu_unsafe(const float *x, float *y, int n) {   // the same ReLU, but without the safety check
    int i = blockIdx.x * blockDim.x + threadIdx.x;   // this thread's global index
    // NO check on y! (x[i % n] keeps the READ safe, so nothing crashes)
    y[i] = fmaxf(x[i % n], 0.0f);
}
'''
unsafe_k = cp.RawKernel(unsafe_src, "relu_unsafe")   # compile it
# 4 blocks of 256 = 1,024 threads for 1,000 elements
n, threads = 1000, 256
blocks = (n + threads - 1) // threads   # = 4 blocks
# all-positive input, and an output with 1,024 extra "sentinel" slots set to −1 after the real data
d_x = cp.random.rand(n, dtype=cp.float32) + 1
d_y = cp.full(n + 1024, -1.0, dtype=cp.float32)   # the output, plus 1,024 sentinel slots set to −1
# launch without the safety check, then count the sentinel slots that are no longer −1
unsafe_k((blocks,), (threads,), (d_x, d_y, cp.int32(n)))
damaged = int((d_y[n:] != -1).sum())   # count the sentinel slots no longer −1 (d_y[n:] = everything after the real data)
print(f"{blocks * threads} threads for {n} elements → {damaged} slots past the end were overwritten!")   # show the damage

*(A 2-D kernel that brightens an image, one thread per pixel, is a Side Trail: Appendix D.2 and the Step 4 Side Trails notebook.)*

### ✏️ YOUR TURN (C)
1. Which block size was slowest? Explain using warps and the 16-blocks-per-SM limit.
2. How many slots were overwritten without the check? Show the arithmetic: 4 × 256 − 1,000 = ?
3. Why did no error message appear? (Section 4.3: the sentinel slots were our own memory.)

*Your answers:*

---
## 🎓 Wrap-up
- Every GPU job does five steps: allocate, copy in, launch, copy out, free. The copies often cost more than the kernel.
- Each thread finds its element from blockIdx, blockDim and threadIdx, and must check `i < n`.
- **Next — Core Lab 9: Keep the GPU Busy.** Asynchronous launches, sync points, and profiling a slow training loop.